In [1]:
import os

In [2]:
%pwd

'd:\\Siam\\End-to-End-ML-Project-with-MLflow\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Siam\\End-to-End-ML-Project-with-MLflow'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories


In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path: Path = CONFIG_FILE_PATH,
        params_file_path: Path = PARAMS_FILE_PATH,
        schema_file_path: Path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config["artifacts_root"]])


    def get_data_ingestion_config(self) -> DataIngestionConfig:

        config = self.config["data_ingestion"]
        create_directories([config["root_dir"]])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config["root_dir"]),
            source_URL=config["source_URL"],
            local_data_file=Path(config["local_data_file"]),
            unzip_dir=Path(config["unzip_dir"])
        )

        return data_ingestion_config

In [8]:
import os
import sys
import urllib.request as request
from zipfile import ZipFile
from mlProject.utils.common import get_size
from mlProject.logger import logging
from mlProject.exception import CustomException

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                self.config.source_URL, self.config.local_data_file
            )
            logging.info(f"{filename} downloaded with following info: \n{headers}")
        else:
            logging.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        '''
        zip_file_path: str
        Extract the zip fine into the data directory
        Function returns None
        '''
        with ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(self.config.unzip_dir)
        os.remove(self.config.local_data_file)  

In [10]:
%pwd

'd:\\Siam\\End-to-End-ML-Project-with-MLflow'

In [11]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    print(data_ingestion_config)

    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise CustomException(e, sys)

[2026-04-10 01:52:42,594: INFO: common: yaml file: D:\Siam\End-to-End-ML-Project-with-MLflow\config\config.yaml loaded successfully]
[2026-04-10 01:52:42,596: INFO: common: yaml file: D:\Siam\End-to-End-ML-Project-with-MLflow\params.yaml loaded successfully]
[2026-04-10 01:52:42,598: INFO: common: yaml file: D:\Siam\End-to-End-ML-Project-with-MLflow\schema.yaml loaded successfully]
[2026-04-10 01:52:42,600: INFO: common: created directory at: artifacts]
[2026-04-10 01:52:42,601: INFO: common: created directory at: artifacts/data_ingestion]
DataIngestionConfig(root_dir=WindowsPath('artifacts/data_ingestion'), source_URL='https://github.com/entbappy/Branching-tutorial/raw/master/winequality-data.zip', local_data_file=WindowsPath('artifacts/data_ingestion/data.zip'), unzip_dir=WindowsPath('artifacts/data_ingestion'))
[2026-04-10 01:52:43,656: INFO: 331746635: artifacts\data_ingestion\data.zip downloaded with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=3